In [2]:
import pandas as pd
import numpy as np
from collections import Counter
df = pd.read_csv("/Users/serhatguldogan/Library/CloudStorage/OneDrive-KocUniversitesi/Project_/RAW DATA/BTC_1sec.csv")


We do the same things that we did at "4_states_transitions_and_means.ipynb". Refer to that.

In [3]:
# Calculate price changes
df['price_change'] = df['midpoint'].diff()

# Function to discretize price changes for 32 states (buckets of 1.25)
def discretize_price_change_32_states(p):
    """
    Discretize price changes into 32 states (16 positive, 16 negative):
    - 0 < p ≤ 1.25 → 1.25
    - 1.25 < p ≤ 2.5 → 2.5
    - ... (continues in 1.25 increments)
    - 18.75 < p ≤ 20 → 20
    Same for negatives (symmetric)
    - 0 stays 0
    """
    bucket_size = 1.25
    max_state = 16  # 16 positive states
    
    if pd.isna(p) or p == 0:
        return 0
    elif p > 0:
        if p > 100:
            return 100  # Cap at 20
        # Find which bucket (1-indexed)
        bucket = int(np.ceil(p / bucket_size))
        if bucket > max_state:
            bucket = max_state
        return bucket * bucket_size
    else:  # p < 0
        if p < -100:
            return -100  # Cap at -20
        # Find which bucket (1-indexed)
        bucket = int(np.ceil(abs(p) / bucket_size))
        if bucket > max_state:
            bucket = max_state
        return -bucket * bucket_size

# Apply discretization
df['price_change_discretized'] = df['price_change'].apply(discretize_price_change_32_states)

# Create label column: map discretized values to labels
# +1.25 → +1, +2.5 → +2, ..., +20 → +16
# -1.25 → -1, -2.5 → -2, ..., -20 → -16
# 0 → 0
def label_price_change_32_states(discretized_value):
    if discretized_value == 0:
        return 0
    elif discretized_value > 0:
        return int(discretized_value / 1.25)
    else:  # discretized_value < 0
        return int(discretized_value / 1.25)

df['price_change_label'] = df['price_change_discretized'].apply(label_price_change_32_states)

# Display results
print("First 20 rows with price changes:")
print(df[['midpoint', 'price_change', 'price_change_discretized', 'price_change_label']].head(20))
print("\nPrice change label distribution:")
print(df['price_change_label'].value_counts().sort_index())


First 20 rows with price changes:
     midpoint  price_change  price_change_discretized  price_change_label
0   56035.995           NaN                      0.00                   0
1   56035.995         0.000                      0.00                   0
2   56035.995         0.000                      0.00                   0
3   56035.995         0.000                      0.00                   0
4   56035.995         0.000                      0.00                   0
5   56035.995         0.000                      0.00                   0
6   56035.995         0.000                      0.00                   0
7   56035.995         0.000                      0.00                   0
8   56035.995         0.000                      0.00                   0
9   56035.995         0.000                      0.00                   0
10  56035.995         0.000                      0.00                   0
11  56035.995         0.000                      0.00                   0
12  

In [4]:
changes = df["price_change_label"]


In [5]:
# Initialize transition counter for 32 states: -16 to -1, 1 to 16
transition_counter = Counter()
for i in range(-16, 0):
    for j in range(-16, 17):
        if j != 0:
            transition_counter[(i, j)] = 0
for i in range(1, 17):
    for j in range(-16, 17):
        if j != 0:
            transition_counter[(i, j)] = 0

# Count transitions
for i in range(1, len(changes)):
    prev_state = changes.iloc[i - 1]
    curr_state = changes.iloc[i]
    # Only count if neither state is 0
    if prev_state != 0 and curr_state != 0:
        transition_counter[(prev_state, curr_state)] += 1

counter = dict(transition_counter)


In [6]:
# Remove transitions involving 0 (shouldn't be any, but just to be safe)
d = counter.copy()
for key in list(counter.keys()):
    if 0 in key:
        d.pop(key)

print(f"Total transitions: {sum(d.values())}")
print(f"Number of transition types: {len(d)}")


Total transitions: 298313
Number of transition types: 1084


In [7]:
# Normalize transition probabilities by first state
normalized_by_first = {}
for a in set(key[0] for key in d.keys()):
    sum_ax = sum(v for (x, y), v in d.items() if x == a)
    if sum_ax == 0:
        continue
    for (x, y), v in d.items():
        if x == a:
            normalized_by_first[(x, y)] = v / sum_ax

print("Sample normalized transitions (first 10):")
for i, (k, v) in enumerate(list(normalized_by_first.items())[:10]):
    print(f"{k}: {v:.6f}")


Sample normalized transitions (first 10):
(1, -16): 0.006324
(1, -15): 0.001121
(1, -14): 0.001256
(1, -13): 0.001547
(1, -12): 0.002130
(1, -11): 0.001727
(1, -10): 0.002781
(1, -9): 0.003879
(1, -8): 0.004350
(1, -7): 0.006951


In [8]:
# Verify normalization (should sum to 1 for each starting state)
print("\nVerification - sums by starting state (first 10):")
for state in sorted(set(key[0] for key in normalized_by_first.keys()))[:10]:
    total = sum(v for (x, y), v in normalized_by_first.items() if x == state)
    print(f"State {state:3d}: {total:.10f}")
print("...")



Verification - sums by starting state (first 10):
State -80: 1.0000000000
State -16: 1.0000000000
State -15: 1.0000000000
State -14: 1.0000000000
State -13: 1.0000000000
State -12: 1.0000000000
State -11: 1.0000000000
State -10: 1.0000000000
State  -9: 1.0000000000
State  -8: 1.0000000000
...


In [9]:
# Create transition matrix (32x32)
# Order: [-16, -15, ..., -1, 1, 2, ..., 16]
indices = list(range(-16, 0)) + list(range(1, 17))
size = len(indices)
mat = np.zeros((size, size))

# Fill matrix
for i, row in enumerate(indices):
    for j, col in enumerate(indices):
        key = (row, col)
        if key in normalized_by_first:
            mat[i, j] = normalized_by_first[key]

print("Transition matrix shape:", mat.shape)
print("\nFirst few rows and columns:")
print(mat[:4, :4])


Transition matrix shape: (32, 32)

First few rows and columns:
[[0.12541632 0.01311407 0.01415487 0.01592423]
 [0.09065934 0.01373626 0.01510989 0.01785714]
 [0.07027027 0.01261261 0.00840841 0.01861862]
 [0.07668554 0.00823469 0.01544004 0.01183736]]


In [10]:
np.savetxt("mat_32_states.csv",X=mat,fmt="%f")

In [11]:
# Solve for stationary distribution: vP = v
# v(P - I) = 0 with constraint sum(v) = 1
P = mat
A = P.T - np.eye(size)

# Append normalization constraint
A = np.vstack([A, np.ones(size)])
b = np.zeros(size + 1)
b[-1] = 1

# Solve using least squares
v, residuals, rank, s = np.linalg.lstsq(A, b, rcond=None)
v = v.real

v= np.array(v)

print("Stationary distribution:")
for i, state in enumerate(indices):
    print(f"State {state:3d}: {v[i]:.6f}")

print(f"\nSum of probabilities: {v.sum():.10f}")


Stationary distribution:
State -16: 0.023539
State -15: 0.003631
State -14: 0.004351
State -13: 0.004908
State -12: 0.005761
State -11: 0.006716
State -10: 0.007919
State  -9: 0.009753
State  -8: 0.011837
State  -7: 0.014832
State  -6: 0.019640
State  -5: 0.027232
State  -4: 0.040041
State  -3: 0.063363
State  -2: 0.087297
State  -1: 0.143815
State   1: 0.167521
State   2: 0.096336
State   3: 0.068901
State   4: 0.044054
State   5: 0.030403
State   6: 0.021985
State   7: 0.016874
State   8: 0.013032
State   9: 0.009954
State  10: 0.008905
State  11: 0.006858
State  12: 0.005727
State  13: 0.004984
State  14: 0.004162
State  15: 0.003590
State  16: 0.022079

Sum of probabilities: 0.9999999983


In [12]:
np.savetxt("pi_32_states.csv",X=v,fmt="%f")

In [13]:
# Calculate mean price change from stationary distribution
# Each state j corresponds to price change = j * 1.25
bucket_size = 1.25

# Calculate mean: E[X] = Σ (state_j * price_change_j * probability_j)
# where price_change_j = state_j * bucket_size
mean_price_change = 0.0

print("State | Price Change | Probability | Contribution")
print("-" * 60)
for i, state in enumerate(indices):
    price_change = state * bucket_size
    prob = v[i]
    contribution = price_change * prob
    mean_price_change += contribution
    if abs(contribution) > 0.001:  # Only print significant contributions
        print(f"{state:5d} | {price_change:11.2f} | {prob:10.6f} | {contribution:12.6f}")

print("-" * 60)
print(f"\nMean price change (from stationary distribution): {mean_price_change:.6f}")


State | Price Change | Probability | Contribution
------------------------------------------------------------
  -16 |      -20.00 |   0.023539 |    -0.470773
  -15 |      -18.75 |   0.003631 |    -0.068080
  -14 |      -17.50 |   0.004351 |    -0.076136
  -13 |      -16.25 |   0.004908 |    -0.079759
  -12 |      -15.00 |   0.005761 |    -0.086421
  -11 |      -13.75 |   0.006716 |    -0.092344
  -10 |      -12.50 |   0.007919 |    -0.098987
   -9 |      -11.25 |   0.009753 |    -0.109719
   -8 |      -10.00 |   0.011837 |    -0.118373
   -7 |       -8.75 |   0.014832 |    -0.129779
   -6 |       -7.50 |   0.019640 |    -0.147300
   -5 |       -6.25 |   0.027232 |    -0.170199
   -4 |       -5.00 |   0.040041 |    -0.200207
   -3 |       -3.75 |   0.063363 |    -0.237612
   -2 |       -2.50 |   0.087297 |    -0.218243
   -1 |       -1.25 |   0.143815 |    -0.179769
    1 |        1.25 |   0.167521 |     0.209401
    2 |        2.50 |   0.096336 |     0.240841
    3 |        3.75 |   0